[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KP-365/Fake_news/blob/main/eval_faithfulness.ipynb)

# Human faithfulness review of generated explanations

This notebook samples 10 committed escalation rows, generates a fresh explanation from displayed structured signals, and creates a CSV template for human review. Use a GPU runtime.

> **Scope note:** `evaluation/escalation_results.csv` preserves labels and NLI verdicts, but not classifier confidence, MC uncertainty, NLI scores, or evidence counts. The notebook therefore recomputes one coherent live signal set from each sampled text snippet and attaches it to that sampled row before calling `explain_decision()`. DDG evidence is live, so these signals are a new faithfulness-review run rather than a reproduction of the historical escalation run.

## 1. Clone the repository and install its pinned environment

The setup is safe to rerun in one Colab session. It installs the repository's versioned requirements rather than selecting package versions in this notebook.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/KP-365/Fake_news.git"
REPO_DIR = Path("/content/Fake_news")

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print(f"Ready in {Path.cwd()}")

## 2. Load the escalation results and select 10 rows

`row_id` is the zero-based row position in the committed CSV. Sampling is without replacement and uses NumPy's generator with seed 42.

In [ ]:
import numpy as np
import pandas as pd

RESULTS_PATH = Path("evaluation/escalation_results.csv")
REQUIRED_COLUMNS = {
    "text_snippet",
    "true_label",
    "classifier_label",
    "verdict",
    "final_label",
}

escalation_results = pd.read_csv(RESULTS_PATH)
missing_columns = REQUIRED_COLUMNS.difference(escalation_results.columns)
if missing_columns:
    raise ValueError(f"Missing escalation columns: {sorted(missing_columns)}")
if len(escalation_results) < 10:
    raise ValueError("At least 10 escalation rows are required")

rng = np.random.default_rng(42)
sample_indices = rng.choice(
    escalation_results.index.to_numpy(), size=10, replace=False
)
sampled_rows = escalation_results.loc[sample_indices].copy()
sampled_rows.insert(0, "row_id", sampled_rows.index.astype(int))
sampled_rows = sampled_rows.reset_index(drop=True)

print("Selected source row IDs:", sampled_rows["row_id"].tolist())
display(
    sampled_rows[[
        "row_id", "text_snippet", "true_label",
        "classifier_label", "verdict", "final_label"
    ]]
)

## 3. Generate and display explanations

The Anthropic key is read with `getpass()`, passed directly to each request, never written to a DataFrame, file, environment variable, or notebook output, and deleted from the notebook namespace in `finally`. For each source row, the exact structured signal dictionary is printed immediately before its explanation.

This cell performs live DDG retrieval and model inference, so outputs and runtime can vary.

In [ ]:
from getpass import getpass
import json
import time

from explain import explain_decision
from pipeline import classify_with_uncertainty
from predict import load_model
from verify import verify_claim

model, tokenizer, device = load_model()
print(f"Classifier startup device: {device}")

def compute_structured_signals(text: str) -> dict:
    classifier_label, confidence, mc_uncertainty = classify_with_uncertainty(
        text, model=model, tokenizer=tokenizer, device=device
    )
    verification = verify_claim(text)
    return {
        "classifier_label": classifier_label,
        "confidence": float(confidence),
        "mc_uncertainty": float(mc_uncertainty),
        "nli_verdict": verification.get("verdict", "insufficient"),
        "max_entailment": float(verification.get("max_entailment", 0.0) or 0.0),
        "max_contradiction": float(verification.get("max_contradiction", 0.0) or 0.0),
        "evidence_count": int(len(verification.get("evidence", []))),
    }

anthropic_key = getpass("Anthropic API key (hidden; held in memory only): " ).strip()
if not anthropic_key:
    raise ValueError("An Anthropic API key is required for this review")

faithfulness_records = []
try:
    for position, row in sampled_rows.iterrows():
        signals = compute_structured_signals(row["text_snippet"])
        explanation = explain_decision(**signals, api_key=anthropic_key)
        record = {
            "row_id": int(row["row_id"]),
            "signals": signals,
            "explanation": explanation,
        }
        faithfulness_records.append(record)

        print(f"\n=== Source row {record['row_id']} ===")
        print("Structured signals passed to explain_decision():")
        print(json.dumps(signals, indent=2))
        print("Generated explanation:")
        print(explanation)

        if position < len(sampled_rows) - 1:
            time.sleep(2)
finally:
    anthropic_key = ""
    del anthropic_key

print(f"\nGenerated {len(faithfulness_records)} explanations.")

## 4. Create the manual review table

The generated explanation is copied into `claim_in_explanation` without automatic interpretation. Review it against the displayed signals, split it into additional claim-level rows when needed, enter `yes` or `no`, and add notes. Rerun the save line after manual edits.

In [ ]:
REVIEW_PATH = Path("evaluation/faithfulness_review.csv")

review_table = pd.DataFrame(
    {
        "row_id": [record["row_id"] for record in faithfulness_records],
        "claim_in_explanation": [
            record["explanation"] for record in faithfulness_records
        ],
        "matches_signals_yes_no": "",
        "notes": "",
    }
)

REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
review_table.to_csv(REVIEW_PATH, index=False)
display(review_table)
print(f"Saved manual review template to {REVIEW_PATH}")